In [ ]:
%run _bootstrap_dev.ipynb

In [ ]:
# New version
def select_top_performing_stocks_NEW(
    data: pd.DataFrame,
    top_percentage: Optional[Union[float, int]] = None,
    *,
    # ---- Selezione primaria ----
    primary: str = "Y",                 # "Y"=annuale, "Q"=trimestrale, "M"=mensile, "R"=rolling mesi
    primary_lookback_months: int = 12,  # usato solo se primary == "R"
    shift_forward: bool = True,         # True = assegna i top del periodo T al periodo T+1
    min_valid_ratio: float = 0.6,       # quota minima di valori non-NaN nel periodo per includere il ticker
    # ---- Selezione secondaria (opzionale) ----
    secondary: Optional[Dict[str, Any]] = None,
    # ---- NUOVO: costruzione Portfolio vectorbt ----
    build_pf: bool = False,
    init_cash: float = 100_000.0,
    freq: str = "1D",
    fee_bps: float = 0.0,               # costo per turnover (bps su notional scambiato)
    return_weights: bool = True         # se True ritorna anche weights_df
) -> Union[
    pd.Series,
    Tuple[pd.Series, "vbt.Portfolio", pd.DataFrame]
]:
    """
    Seleziona i top performer su base annuale, trimestrale, mensile o rolling n-mesi.
    Supporta un secondo filtro opzionale (rolling breve) tra i candidati del filtro primario.

    (NUOVO) Se build_pf=True, costruisce un portafoglio equal-weight "buy & hold per periodo di applicazione"
    e restituisce un vectorbt.Portfolio ottenuto da returns (senza doppia rotazione / senza WFO).

    Returns
    -------
    - Se build_pf=False:
        prim_lists : Series {periodo_applicazione -> [tickers]}
    - Se build_pf=True:
        (prim_lists, pf, weights_df)  (weights_df opzionale via return_weights)
    """

    # ---------------------------
    # Helper interni
    # ---------------------------
    def _ensure_datetime_index(df: pd.DataFrame) -> pd.DataFrame:
        if not isinstance(df.index, pd.DatetimeIndex):
            df = df.copy()
            df.index = pd.to_datetime(df.index)
        return df

    def _normalize_close(df: pd.DataFrame) -> pd.DataFrame:
        # Gestione MultiIndex con livello "Close"
        if isinstance(df.columns, pd.MultiIndex):
            if "Close" in df.columns.get_level_values(0):
                df = df.xs("Close", level=0, axis=1)
            elif "Close" in df.columns.get_level_values(-1):
                df = df.xs("Close", level=-1, axis=1)
            else:
                raise ValueError("Impossibile trovare 'Close' nelle colonne MultiIndex.")
        return df

    def _percent_valid(s: pd.Series) -> float:
        n = len(s)
        return float(s.notna().sum()) / float(n) if n > 0 else 0.0

    def _resample_returns_first_last(df: pd.DataFrame, rule: str) -> pd.DataFrame:
        """Rendimento per periodo: last/first - 1 su resample."""
        g_first = df.resample(rule).first()
        g_last  = df.resample(rule).last()
        out = (g_last / g_first) - 1.0
        return out

    def _apply_valid_ratio_mask(df: pd.DataFrame, rule: str, ratio: float) -> pd.DataFrame:
        """Maschera colonne periodo-per-periodo se non rispettano min_valid_ratio nel periodo."""
        masked = []
        for end_ts, frame in df.resample(rule):
            keep = frame.apply(_percent_valid, axis=0) >= float(ratio)
            if keep.any():
                r = _resample_returns_first_last(frame.loc[:, keep], rule).iloc[-1:]
                masked.append(r)
            else:
                masked.append(pd.DataFrame(index=[end_ts], columns=df.columns))
        res = pd.concat(masked).sort_index()
        return res

    def _rolling_monthly_total_return(df: pd.DataFrame, months: int) -> pd.DataFrame:
        """Total return su base mensile: (P_t / P_{t-months}) - 1 calcolato ai month-end."""
        m = df.resample("ME").last()
        return (m / m.shift(months)) - 1.0

    def _top_k_from_rank_row(rank_row: pd.Series, top) -> list:
        rr = rank_row.dropna()
        if rr.empty:
            return []
        if top is None:
            return list(rr.index)
        if isinstance(top, float):
            k = max(1, int(np.floor(len(rr) * top)))
        else:
            k = int(top)
        k = max(1, min(k, len(rr)))
        return list(rr.sort_values(ascending=True).index[:k])  # rank più basso = migliore

    def _period_bounds_from_label(label, primary: str):
        """Restituisce (start_ts, end_ts) inclusivi per periodo di APPLICAZIONE."""
        if primary == "Y":
            y = int(label)
            return pd.Timestamp(year=y, month=1, day=1), pd.Timestamp(year=y, month=12, day=31)
        elif primary == "Q":
            p = pd.Period(label, freq="Q")
            return p.start_time.normalize(), p.end_time.normalize()
        else:  # "M" o "R"
            p = pd.Period(label, freq="M")
            return p.start_time.normalize(), p.end_time.normalize()

    # ---------------------------
    # Normalizzazione input
    # ---------------------------
    df = _ensure_datetime_index(data)
    df = _normalize_close(df)
    df = df.sort_index()

    # ---------------------------
    # 1) Calcolo dei ritorni primari per periodo
    # ---------------------------
    if primary == "Y":
        primary_returns = _apply_valid_ratio_mask(df, "YE", min_valid_ratio)
        selection_index = primary_returns.index.year.astype(int)
    elif primary == "Q":
        primary_returns = _apply_valid_ratio_mask(df, "QE", min_valid_ratio)
        selection_index = primary_returns.index.to_period("Q").astype(str)
    elif primary == "M":
        primary_returns = _apply_valid_ratio_mask(df, "ME", min_valid_ratio)
        selection_index = primary_returns.index.to_period("M").astype(str)
    elif primary == "R":
        rets = _rolling_monthly_total_return(df, primary_lookback_months)
        m_last = df.resample("ME").last()
        valid_count = m_last.notna().astype(int).rolling(primary_lookback_months, min_periods=1).sum()
        need = int(np.ceil(primary_lookback_months * min_valid_ratio))
        primary_returns = rets.where(valid_count >= need, np.nan).dropna(how="all")
        selection_index = primary_returns.index.to_period("M").astype(str)
    else:
        raise ValueError("primary deve essere uno tra {'Y','Q','M','R'}")

    if primary_returns.empty:
        prim_lists = pd.Series(dtype=object, name="Top_Tickers")
        return prim_lists if not build_pf else (prim_lists, vbt.Portfolio.from_returns(pd.Series(dtype=float), init_cash=init_cash, freq=freq), pd.DataFrame())

    ranked = primary_returns.rank(axis=1, ascending=False, method="first")

    # ---------------------------
    # 2) Selezione primaria
    # ---------------------------
    if top_percentage is None:
        prim_lists_sel = primary_returns.apply(lambda row: list(row.index[row.notna()]), axis=1)
    else:
        prim_lists_sel = ranked.apply(lambda row: _top_k_from_rank_row(row, top_percentage), axis=1)

    prim_lists_sel.index = selection_index  # periodi di SELEZIONE

    # ---------------------------
    # 3) Selezione secondaria (opzionale)
    # ---------------------------
    if secondary is not None:
        sec_n = int(secondary.get("lookback_months", 0) or 0)
        sec_top = secondary.get("top", None)
        if sec_n <= 0:
            raise ValueError("secondary.lookback_months deve essere un intero > 0")

        short_rets = _rolling_monthly_total_return(df, sec_n).sort_index()  # ai month-end (ME)

        def _closest_me_le(idx: pd.DatetimeIndex, ref: pd.Timestamp) -> Optional[pd.Timestamp]:
            pos = idx.searchsorted(ref, side="right") - 1
            return None if pos < 0 else idx[pos]

        refined = []
        for idx_lab, candidates in prim_lists_sel.items():
            if not candidates:
                refined.append([])
                continue

            if primary == "Y":
                ref_end = pd.Timestamp(year=int(idx_lab), month=12, day=31)
                ref_me = ref_end.to_period("M").to_timestamp("M")
            elif primary == "Q":
                ref_me = pd.Period(idx_lab, freq="Q").end_time.to_period("M").to_timestamp("M")
            else:
                ref_me = pd.Period(idx_lab, freq="M").to_timestamp("M")

            use_me = _closest_me_le(short_rets.index, ref_me)
            if use_me is None:
                refined.append(candidates)
                continue

            cands = [t for t in candidates if t in short_rets.columns]
            if not cands:
                refined.append([])
                continue

            row_short = short_rets.loc[use_me, cands].dropna()
            if row_short.empty:
                refined.append([])
                continue

            rank_short = (-row_short).rank(ascending=True, method="first")
            refined.append(_top_k_from_rank_row(rank_short, sec_top))

        prim_lists_sel = pd.Series(refined, index=prim_lists_sel.index, dtype=object)

    # ---------------------------
    # 4) Shift al periodo successivo (APPLICAZIONE)
    # ---------------------------
    if shift_forward:
        if primary == "Y":
            new_index = (pd.Series(prim_lists_sel.index, dtype=int) + 1).astype(int)
            prim_lists = pd.Series(prim_lists_sel.values, index=new_index, dtype=object)
        elif primary == "Q":
            next_idx = (pd.PeriodIndex(prim_lists_sel.index, freq="Q") + 1).astype(str)
            prim_lists = pd.Series(prim_lists_sel.values, index=next_idx, dtype=object)
        else:
            next_idx = (pd.PeriodIndex(prim_lists_sel.index, freq="M") + 1).astype(str)
            prim_lists = pd.Series(prim_lists_sel.values, index=next_idx, dtype=object)
    else:
        prim_lists = prim_lists_sel.copy()

    prim_lists.name = "Top_Tickers"

    # ---------------------------
    # 5) (NUOVO) Build Portfolio vectorbt
    # ---------------------------
    if not build_pf:
        return prim_lists

    prices = df.copy()
    # daily returns (no look-ahead)
    asset_ret = prices.pct_change()

    # weights daily (piecewise constant per periodo di applicazione)
    weights = pd.DataFrame(0.0, index=prices.index, columns=prices.columns)

    # costruisci bounds per periodo di applicazione
    bounds = []
    for lab in prim_lists.index:
        start, end = _period_bounds_from_label(lab, primary)
        bounds.append((lab, start, end))

    for lab, start_ts, end_ts in bounds:
        sel = prim_lists.loc[lab]
        if not sel:
            continue
        sel = [t for t in sel if t in weights.columns]
        if not sel:
            continue

        # limita a date disponibili
        idx_slice = weights.loc[start_ts:end_ts].index
        if len(idx_slice) == 0:
            continue

        w = 1.0 / float(len(sel))
        weights.loc[idx_slice, sel] = w

    # Portfolio returns: usa weights della barra precedente (execution delay 1)
    w_prev = weights.shift(1).fillna(0.0)
    port_ret = (w_prev * asset_ret).sum(axis=1)

    # costi su turnover ai rebalance days (half-turnover)
    if fee_bps and fee_bps != 0.0:
        turnover = (weights - weights.shift(1)).abs().sum(axis=1) / 2.0
        cost = turnover * (float(fee_bps) / 1e4)
        port_ret = port_ret - cost

    port_ret = port_ret.fillna(0.0)
    price_curve = (1 + port_ret).cumprod() * init_cash
    pf = vbt.Portfolio.from_holding(
        price_curve.rename("asset").to_frame(),
        init_cash=float(init_cash),
        freq=freq
    )
    # pf = vbt.Portfolio.from_returns(
    #     port_ret,
    #     init_cash=float(init_cash),
    #     freq=freq
    # )

    if return_weights:
        return prim_lists, pf, weights
    return prim_lists, pf, pd.DataFrame()

In [ ]:
us_megastocks = ["AAPL","MSFT","GOOGL","AMZN","META","NVDA","JPM","JNJ","XOM","WMT"]

In [ ]:
tickers=stocks_euro
# tickers=ai_dc_quantum_thematic
# tickers=quantum_pureplays

init_cash=100_000
normalize=False # Nei rotazionali va lasciato False -> un titolo non compare nelle selezizoni dove ha NaN
lookback_buffer=365

start_date="2015-01-01"
end_date=None
# normalize=True
show_progress=False

download_start_date = (pd.to_datetime(start_date) - timedelta(days=lookback_buffer)).strftime("%Y-%m-%d")
stocks_data, company_data = fetch_data_and_companies(tickers, download_start_date, end_date,normalize=normalize)


In [ ]:
company_data

In [ ]:
stocks_data.head()

In [ ]:
stocks_data.tail()

In [ ]:
# top_true  = select_top_performing_stocks(data, top_percentage=0.2, primary="Y", shift_forward=True)
# top_false = select_top_performing_stocks(data, top_percentage=0.2, primary="Y", shift_forward=False)

# print(top_false.index[:3], "→", top_true.index[:3])   # indici diversi
# print(top_false.iloc[0] == top_true.iloc[1])          # liste identiche, ma “slittate” di 1


In [ ]:
# close = data.Close
# 1) Top annuali (come ora), assegnati all’anno successivo:
top_yr = select_top_performing_stocks(stocks_data, top_percentage=0.2, primary="Y", shift_forward=True)
# -> indice: 2016, 2017, ..., valori: liste di ticker
top_yr

In [ ]:
# top_yr_no_shift = select_top_performing_stocks(close, top_percentage=0.2, primary="Y", shift_forward=False)
# # -> indice: 2016, 2017, ..., valori: liste di ticker
# top_yr_no_shift

In [ ]:
# 2) Top trimestrali (top 5 fissi), assegnati al trimestre successivo:
top_q = select_top_performing_stocks(stocks_data, top_percentage=5, primary="Q", shift_forward=True)
# -> indice: "2024Q2", "2024Q3", ...
top_q

In [ ]:
# 3) Rolling 12 mesi (tipo momentum), selezione mensile, poi filtro secondario 3 mesi:
# top_roll = select_top_performing_stocks(
#     stocks_data,
#     top_percentage=0.3,        # primi 30% per rolling 12m
#     primary="Y",
#     primary_lookback_months=12,
#     shift_forward=True,
#     secondary={"lookback_months": 3, "top": 0.5}  # metà migliori su 3m tra i candidati 12m
# )
# # -> indice: "YYYY-MM" del mese SUCCESSIVO
# my_display(top_roll)

# Nuovo selettiore con calcolo delle performance di portafoglio

# Selezione primaria (obbligatoria)
primary="R"
top_percentage=10                    # i primi top_percentage nell'ultimo primary periodo
primary_lookback_months=12           # considerato solo se primary="R" (rolling)
shift_forward=True                   # True -> no look-ahead bias. Obbligatorio per real trade e calcolo performance!

# Selezione opzionale secondaria (dual momentum):
lookback_months=4
n_top=5                              # i primi n negli ultimi lookback_months mesi
secondary={"lookback_months": lookback_months, "top": n_top}  # selettore secondario (dual momentum): i primi ntop negli ultimi lookback_months tra i candidati selezionai dal metodo primary

# Calcolo performance (opzionale): se true ritorna 3 valori, altrimenti 1
compute_metrics = True
weighting = "equal"           # "equal" (per ora)
fee_bps = 0.01                # costo di ribilanciamento per periodo (in basis points, es. 5 = 0.05%)
rf = 0.02                     # tasso risk-free ANNUO (decimale) per Sharpe/Sortino

# prim_lists, *_ = select_top_performing_stocks(
prim_lists, pf_metrics, pf_frame = select_top_performing_stocks(
    stocks_data,
    primary=primary,                         # Selezione primaria: gestisce la frequenza di ribilanciamento
    top_percentage=top_percentage,       # primi top_percentage titoli nell'ultimo primary 
    primary_lookback_months=primary_lookback_months,          # considerato solo se primary="R" (rolling)
    shift_forward=shift_forward,                  # True -> no look-ahead bias
    secondary=secondary,
    compute_metrics=compute_metrics,
    weighting=weighting,                
    fee_bps=fee_bps,                  
    rf = rf                             
)


# 1) Tabella “selezione per periodo”
my_display(prim_lists, "Top selezionati per periodo")  # OK (DataFrame)

if compute_metrics:     
    # 2) Tabella rendimenti
    my_display(pf_frame, "Rendimenti per periodo")        # OK (DataFrame)
    # 3) Tabella metriche
    metrics_df = pd.DataFrame([pf_metrics]).T.rename(columns={0: "value"})
    my_display(metrics_df, "Metriche portafoglio")        # OK (DataFrame)


In [ ]:
build_pf = True
return_weights = True         # se True ritorna anche weights_df


prim_lists, pf_rot, pf_frame = select_top_performing_stocks_NEW(
    stocks_data,
    primary=primary,                         # Selezione primaria: gestisce la frequenza di ribilanciamento
    top_percentage=top_percentage,       # primi top_percentage titoli nell'ultimo primary 
    primary_lookback_months=primary_lookback_months,          # considerato solo se primary="R" (rolling)
    shift_forward=shift_forward,                  # True -> no look-ahead bias
    secondary=secondary,
    fee_bps=fee_bps,    
    build_pf=build_pf,
    return_weights=return_weights
)


In [ ]:
prim_lists

In [ ]:
pf_rot.stats()

In [ ]:
# # 4) Universe completo per periodo (nessun filtro), ma poi riduci coi 6 mesi:

# full_then_6m = select_top_performing_stocks(
#     data,
#     top_percentage=None,             # nessun taglio primario
#     primary="Y",
#     shift_forward=True,
#     secondary={"lookback_months": 6, "top": 3}  # tieni i migliori 3 su 6 mesi
# )
# full_then_6m

In [ ]:
# tickers = stocks_euro
# data = download_data(tickers,start_date="2015-01-01", end_date="2025-01-01")
# top_tickers=select_top_performing_stocks(data,0.3) 
# top_tickers[2025]

In [ ]:
# weights_2025 = build_soft_weights_for_year(
#     data, year=2025,
#     k_target=25,
#     primary_top=0.3,                         # fai in modo che il primario contenga >= 25 titoli
#     secondary={"lookback_months": 3, "top": None},  # nessun taglio: il 3m influenza i pesi (non il count)
#     alpha=0.5, beta=1.0                      # “solo primario” pesa metà di “confermato”
# )
# weights_2025

## Universe momentum WFO

In [ ]:
param_grid = {
    "primary": ["Y", "M", "R"],
    "top_percentage": [5, 10, 15],            # primi k (numeri interi)
    "primary_lookback_months": [6, 9, 12],    # usato se primary="R"
    "secondary_lookback_months": [3, 4, 6],
    "secondary_top": [3, 5, 8]
}

summary_df, results_df, oos_equity, winners_params = wfo_universe_selector_momentum(
    data=stocks_data,                  # df prezzi wide (Close adjusted)
    train_years=3,
    test_years=1,
    start_year=2018,                   # opzionale; se None calcola dai dati
    end_year=2025,                     # opzionale
    primary="R",
    param_grid=param_grid,
    min_valid_ratio=0.6,
    shift_forward=True,                # fondamentale per evitare look-ahead
    compute_metrics=True,              # deve rimanere True
    weighting="equal",
    fee_bps=0.01,                      # 0.01 bps per periodo (esempio)
    rf=0.02,                           # 2% annuo
    selection_rule="composite",        # ranking multimetrico
    selection_weights=None,            # default pesi
    verbose=True
)

In [ ]:
# winners_params

In [ ]:
# Visualizza
my_display(summary_df, "WFO – Best per blocco")
# my_display(results_df, "WFO – Tutte le combo (per blocco)")
my_display(oos_equity.to_frame("equity"), "WFO – Equity OOS")

## Strategy WFO

In [ ]:
# 1) walk-forward con universo fisso
# tickers=us_megastocks
# tickers=list(weights_2025.index)

rets, summary, weights = wfo_method_rotation_allocation(
    tickers=tickers,
    start="2015-01-01", end="2025-01-01",
    train_years=3, test_years=1,
    rf=0.0, tc=0.001,
    rebalance_freqs=["ME","QE","YE","BH"],   # puoi ridurre a ["ME","QE"]
    min_valid_ratio=0.9,
    verbose=True
)

# 2) verifica che non ci siano buchi/duplicati
assert rets.index.is_monotonic_increasing and rets.index.is_unique


In [ ]:
# print WFO results
print_walkforward_summary(rets, summary, weights, title="📈 Performance aggregata (walk-forward)")

In [ ]:
# Print trading plain 
plan = generate_portfolio_plan(summary, weights, capitale=10_000,max_titoli=None)
my_display(plan)

In [ ]:
# %run u_functions.ipynb
# o = load_ohlcv(benchmark_ticker).Close

In [ ]:
portfolios_returns = {"(WFO Strategies)": rets}
benchmark_ticker = 'SPY'
fig_vs = plot_multiple_portfolios(portfolios_returns, benchmark=benchmark_ticker)
fig_vs.show()

plot_weights_heatmap(weights, title="Allocazioni medie (Walk-Forward)")
plot_strategy_pie(summary, title="Distribuzione strategie selezionate")